# Melanoma & Nevus Dermoscopy Dataset - Imputation (Stratified Random Hot-Deck)

##### 1. Load dữ liệu train/val/test

In [1]:
import pandas as pd
import numpy as np

df_train = pd.read_csv("data/splitted/train.csv")
df_val = pd.read_csv("data/splitted/val.csv")
df_test = pd.read_csv("data/splitted/test.csv")

print("Train:", df_train.shape)
print("Val  :", df_val.shape)
print("Test :", df_test.shape)

Train: (12696, 10)
Val  : (2752, 10)
Test : (2685, 10)


##### 2. Gộp nhóm attribution

Áp dụng cùng mapping đã dùng ở `02_EDA.ipynb` cho cả 3 tập (train/val/test), để dùng `attribution_grouped` làm điều kiện impute.

In [2]:
attribution_group_map = {
    'ViDIR Group, Department of Dermatology, Medical University of Vienna': 'Department of Dermatology, Medical University of Vienna',
    'Hospital Clínic de Barcelona': 'Hospital Clinic de Barcelona',
    'Department of Dermatology, Hospital Clínic de Barcelona': 'Hospital Clinic de Barcelona',
    'Sydney Melanoma Diagnostic Center at Royal Prince Alfred Hospital, Pascale Guitera': 'Royal Prince Alfred Hospital',
    'Sydney Melanoma Diagnostic Center at Royal Prince Alfred Hospital': 'Royal Prince Alfred Hospital',
    'Anonymous': 'Other Sources',
    'Dr. Julien Anriot': 'Other Sources',
}

for df_ in [df_train, df_val, df_test]:
    df_['attribution_grouped'] = df_['attribution'].replace(attribution_group_map)

df_train['attribution_grouped'].value_counts()

attribution_grouped
Hospital Clinic de Barcelona                                                                                    4098
Other Sources                                                                                                   3500
Department of Dermatology, Medical University of Vienna                                                         3000
Memorial Sloan Kettering Cancer Center                                                                          1435
Royal Prince Alfred Hospital                                                                                     358
Hospital Italiano de Buenos Aires                                                                                250
The University of Queensland Diamantina Institute, The University of Queensland, Dermatology Research Centre      55
Name: count, dtype: int64

##### 3. Xây dựng pool impute (chỉ từ `df_train`)

Pool = tập giá trị thật (không missing) dùng để random sample khi fill. Chỉ xây từ `df_train` để tránh leakage, sau đó dùng chung pool này cho cả train/val/test.

- `age_approx`, `sex`: pool theo `(diagnosis, attribution_grouped)`, fallback `attribution_grouped`, fallback cuối global.
- `anatom_site_general`: pool theo `attribution_grouped` only, fallback global.

In [3]:
def build_pool_2level(df, col, col_a, col_b):
    """Pool theo (col_a, col_b), dict {(val_a, val_b): array giá trị thật}."""
    non_missing = df[df[col].notna()]
    return non_missing.groupby([col_a, col_b])[col].apply(lambda s: s.values).to_dict()


def build_pool_1level(df, col, col_a):
    """Pool theo col_a, dict {val_a: array giá trị thật}."""
    non_missing = df[df[col].notna()]
    return non_missing.groupby(col_a)[col].apply(lambda s: s.values).to_dict()


# --- age_approx ---
age_pool_diag_attr = build_pool_2level(df_train, 'age_approx', 'diagnosis', 'attribution_grouped')
age_pool_attr = build_pool_1level(df_train, 'age_approx', 'attribution_grouped')
age_pool_global = df_train['age_approx'].dropna().values

# --- sex ---
sex_pool_diag_attr = build_pool_2level(df_train, 'sex', 'diagnosis', 'attribution_grouped')
sex_pool_attr = build_pool_1level(df_train, 'sex', 'attribution_grouped')
sex_pool_global = df_train['sex'].dropna().values

# --- anatom_site_general ---
anatom_pool_attr = build_pool_1level(df_train, 'anatom_site_general', 'attribution_grouped')
anatom_pool_global = df_train['anatom_site_general'].dropna().values

print("Số nhóm pool (diagnosis, attribution_grouped) cho age:", len(age_pool_diag_attr))
print("Số nhóm pool attribution_grouped cho anatom:", len(anatom_pool_attr))

Số nhóm pool (diagnosis, attribution_grouped) cho age: 14
Số nhóm pool attribution_grouped cho anatom: 7


##### 4. Hàm impute (stratified random hot-deck, có fallback)

In [4]:
rng = np.random.default_rng(42)


def impute_2level(df, col, col_a, col_b, pool_full, pool_fallback, pool_global, rng):
    """Fill NaN trong df[col] bằng random hot-deck, group theo (col_a, col_b),
    fallback col_b, fallback global."""
    result = df[col].copy()
    missing_idx = df.index[result.isna()]

    for idx in missing_idx:
        key_full = (df.loc[idx, col_a], df.loc[idx, col_b])
        pool = pool_full.get(key_full)
        if pool is None or len(pool) == 0:
            pool = pool_fallback.get(df.loc[idx, col_b])
        if pool is None or len(pool) == 0:
            pool = pool_global
        result.loc[idx] = rng.choice(pool)

    return result


def impute_1level(df, col, col_a, pool_full, pool_global, rng):
    """Fill NaN trong df[col] bằng random hot-deck, group theo col_a,
    fallback global."""
    result = df[col].copy()
    missing_idx = df.index[result.isna()]

    for idx in missing_idx:
        pool = pool_full.get(df.loc[idx, col_a])
        if pool is None or len(pool) == 0:
            pool = pool_global
        result.loc[idx] = rng.choice(pool)

    return result

##### 5. Áp dụng impute cho train/val/test

In [5]:
print("Missing trước khi impute:")
for name, d in [('train', df_train), ('val', df_val), ('test', df_test)]:
    print(f"  {name}:", d[['age_approx', 'sex', 'anatom_site_general']].isna().sum().to_dict())

for d in [df_train, df_val, df_test]:
    d['age_approx'] = impute_2level(
        d, 'age_approx', 'diagnosis', 'attribution_grouped',
        age_pool_diag_attr, age_pool_attr, age_pool_global, rng,
    )
    d['sex'] = impute_2level(
        d, 'sex', 'diagnosis', 'attribution_grouped',
        sex_pool_diag_attr, sex_pool_attr, sex_pool_global, rng,
    )
    d['anatom_site_general'] = impute_1level(
        d, 'anatom_site_general', 'attribution_grouped',
        anatom_pool_attr, anatom_pool_global, rng,
    )

print("\nMissing sau khi impute:")
for name, d in [('train', df_train), ('val', df_val), ('test', df_test)]:
    print(f"  {name}:", d[['age_approx', 'sex', 'anatom_site_general']].isna().sum().to_dict())

Missing trước khi impute:
  train: {'age_approx': 324, 'sex': 274, 'anatom_site_general': 785}
  val: {'age_approx': 73, 'sex': 77, 'anatom_site_general': 191}
  test: {'age_approx': 61, 'sex': 46, 'anatom_site_general': 159}



Missing sau khi impute:
  train: {'age_approx': 0, 'sex': 0, 'anatom_site_general': 0}
  val: {'age_approx': 0, 'sex': 0, 'anatom_site_general': 0}
  test: {'age_approx': 0, 'sex': 0, 'anatom_site_general': 0}
